# Exploratory load-total calibration

This notebook applies `src/ausgrid_synth/calibration.py` to existing `diffusion_v2_post` samples. Each seed's empirical daily GC+CL mapping is fitted only on generated and observed **training-household** contexts. The same map is applied to test and privacy contexts; solar values and the night mask are unchanged. The previous experiments already inspected the test set, so use these results as exploratory evidence and confirm any method choice on a future untouched household split. No GPU or retraining is needed.

## 1. Locate prepared data and source samples

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, cwd.parent, cwd/'ausgrid-synthetic',
              Path('/content/drive/MyDrive/ausgrid-synthetic/ausgrid-synthetic'),
              Path('/content/drive/MyDrive/ausgrid-synthetic'))
PROJECT_ROOT = next((p for p in candidates if (p/'src/ausgrid_synth/cli.py').is_file()
                     and (p/'data/prepared/prepared.npz').is_file()), None)
if PROJECT_ROOT is None and 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = next((p for p in candidates if (p/'src/ausgrid_synth/cli.py').is_file()
                         and (p/'data/prepared/prepared.npz').is_file()), None)
assert PROJECT_ROOT is not None, 'Locate the project with src/ and prepared.npz, or run the preparation notebook first.'
os.environ['PYTHONPATH'] = str(PROJECT_ROOT/'src')
if os.environ['PYTHONPATH'] not in sys.path: sys.path.insert(0,os.environ['PYTHONPATH'])
from ausgrid_synth.data import load_prepared
from ausgrid_synth.cli import context_indices
DATA = load_prepared(PROJECT_ROOT/'data/prepared/prepared.npz')
SEEDS = (1,2,3)
REPORTS = PROJECT_ROOT/'outputs/reports'
SAMPLES = PROJECT_ROOT/'outputs/samples'
print('Project:', PROJECT_ROOT, '| split households:',
      {name:len(np.unique(DATA['customer'][DATA['split']==code]))
       for code,name in enumerate(('train','validation','test'))})

def run(stage, arm, seed=1, **options):
    cmd=[sys.executable,'-m','ausgrid_synth.cli',stage,'--root',str(PROJECT_ROOT),
         '--arm',arm,'--seed',str(seed)]
    for key,value in options.items(): cmd.extend(['--'+key.replace('_','-'),str(value)])
    subprocess.run(cmd,check=True,env=os.environ.copy())
for seed in SEEDS:
    for context in ('train','test','privacy'):
        assert (SAMPLES/f'diffusion_v2_post_seed{seed}_{context}.npz').is_file(), (seed,context)


## 2. Fit training-only mappings and write calibrated samples

The source files, mapping, and calibrated files have checked provenance. Reruns verify existing outputs; a changed experiment requires a new arm name.

In [ ]:
for seed in SEEDS: run('calibrate','diffusion_v2_loadcal_post',seed=seed)


## 3. Evaluate and compare all three seeds

In [ ]:
for seed in SEEDS: run('evaluate','diffusion_v2_loadcal_post',seed=seed)
def report_table(arms):
    rows=[]
    for arm in arms:
        for seed in SEEDS:
            path=REPORTS/f'{arm}_seed{seed}.json'
            if not path.exists(): continue
            r=json.loads(path.read_text()); f=r['fidelity_and_physics']; u=r['utility']; p=r['disclosure_probe']
            rows.append({'arm':arm,'seed':seed,
                         'load_daily_W1':f['total_consumption_daily_kwh_wasserstein'],
                         'load_peak_W1':f['total_consumption_peak_interval_kwh_wasserstein'],
                         'solar_daily_W1':f['gross_generation_daily_kwh_wasserstein'],
                         'night_GG_kWh':f['night_solar_kwh'],
                         'synthetic_to_real_MAE':u['synthetic_train_test_real_mae'],
                         'real_to_real_MAE':u['real_train_test_real_mae'],
                         'attack_AUC':p['household_membership_auc']})
    table=pd.DataFrame(rows)
    display(table.round(4))
    if len(table): display(table.groupby('arm').mean(numeric_only=True).drop(columns='seed').round(4))
    return table
results=report_table(('vae_post','diffusion_v2_post','diffusion_v2_loadcal_post'))
base=results[results.arm=='diffusion_v2_post'].set_index('seed').select_dtypes('number')
cal=results[results.arm=='diffusion_v2_loadcal_post'].set_index('seed').select_dtypes('number')
assert len(base)==len(cal)==3
print('Calibrated minus base (negative W1/MAE is better):')
display((cal-base).round(4))
assert np.allclose(cal['solar_daily_W1'],base['solar_daily_W1'])
assert np.allclose(cal['night_GG_kWh'],base['night_GG_kWh'])


## Interpretation

Global daily-total calibration can improve the load marginal while changing peak energy, conditional subgroups, load–solar dependence, utility, and the measured attack. Inspect the saved reports across seeds. The mapping is an optional postprocessing arm, not a new model or a privacy guarantee.